In [ ]:
from web3 import Web3
import json
import os

infura_key = ''
wallet_public_address = Web3.to_checksum_address('')
wallet_private_key = ''

USDC_address = Web3.to_checksum_address('0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8')
USDT_address = Web3.to_checksum_address('0xaA8E23Fb1079EA71e0a56F48a2aA51851D8433D0') 
USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))
print("Connected to Sepolia Testnet:", web3)
abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")

USDT_contract = web3.eth.contract(address=USDT_address, abi=abi_data['ERC20_ABI'])
USTUSD_contract = web3.eth.contract(address=USTUSD_address, abi=abi_data['ERC20_ABI'])

Connected to Sepolia Testnet: <web3.main.Web3 object at 0x000001675F571DB0>
ABI loaded successfully.


## Check the current price of USDT in USTUSD from Uniswap V3 Pool.

In [31]:
factory_addr = '0x0227628f3F023bb0B980b67D528571c95c6DaC1c'
factory_contract = web3.eth.contract(factory_addr, abi=abi_data['UNISWAP_FACTORY_ABI'])
def get_pool_address(tokenA, tokenB, tier_fee, factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    # tier_fee: 100 for 0.01%, 500 for 0.05%, 3000 for 0.3%, 10000 for 1%
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, tier_fee).call()
    return pool_address

USDT_USTUSD_pool_address = get_pool_address(USTUSD_address, USDT_address, 500, factory_contract)
print("USDT-USTUSD Pool Address:", USDT_USTUSD_pool_address)

USDT_USTUSD_pool = web3.eth.contract(address=USDT_USTUSD_pool_address, abi=abi_data['UNISWAP_V3_POOL_ABI'])

USDT_price_in_USTUSD = USDT_USTUSD_pool.functions.slot0().call()[0]**2 / 2**192 / (10**12)
print(f"1 USDT = {USDT_price_in_USTUSD:.6f} USTUSD")

USDT-USTUSD Pool Address: 0x49e75DCCCf6Bb59531dC52Ea85579Bc460A59ccF
1 USDT = 4.162606 USTUSD


## Perform the swap on Uniswap V3 Pool

- original number of USDT in the pool: 51517.079575
- original number of USTUSD in the pool: 412100.796150338963559755

In [28]:
# approve the Universal Router to spend USDT on behalf of the wallet
universal_router_address = web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')
universal_router_contract = web3.eth.contract(address=universal_router_address, abi=abi_data['UNIVERSAL_ROUTER_ABI'])
permit2_address = web3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
permit2_contract = web3.eth.contract(address=permit2_address, abi=abi_data['PERMIT2_ABI'])

def send_tx(tx, private_key):
    signed_txn = web3.eth.account.sign_transaction(tx, private_key)
    raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
    tx_hash = web3.eth.send_raw_transaction(raw_transaction)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    print(f"Transaction sent: {tx_hash.hex()} ", end=" ; ")
    receipt_status = "success" if receipt.status == 1 else "failure"
    print(f"Transaction Status: {receipt_status}") 
    return tx_hash, receipt

def approve_token_spending(web3, contract, wallet_public_address, *approve_args):
    tx = contract.functions.approve(*approve_args).build_transaction({
        "from": wallet_public_address,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
    })
    tx_hash, receipt = send_tx(tx, wallet_private_key)


approve_token_spending(web3, USDT_contract, wallet_public_address, permit2_address, 2**256 - 1)
approve_token_spending(web3, permit2_contract, wallet_public_address, USDT_address, universal_router_address, 2**160 - 1, 2**48 - 1)
approve_token_spending(web3, USTUSD_contract, wallet_public_address, permit2_address, 2**256 - 1)
approve_token_spending(web3, permit2_contract, wallet_public_address, USTUSD_address, universal_router_address, 2**160 - 1, 2**48 - 1)

Transaction sent: 9416737a7f10a899acf21de768a0df347fb233ea4b1871a413ccaa338504f285  ; Transaction Status: success
Transaction sent: 4ca7a16ef8827f7887c43df487c61ac1824215f79c2c2bb2d08a913173c03652  ; Transaction Status: success
Transaction sent: d398e6112516976e8032d779d58dc4d54ae73a2150028948b2f7568f1ca041bc  ; Transaction Status: success
Transaction sent: 8d408f2415efc2edf99ff34b8471c4f5e11711ed093dea2aca26daa32401a773  ; Transaction Status: success


In [30]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
codec = RouterCodec()

USDT_in_amount = 20000 * 10**6  # 

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        USDT_in_amount,  # amount in, (20000 USDT with 6 decimals)
        0,
        [   
            USDT_address,
            500,
            USTUSD_address
        ],
    ).build(2**256 - 1)

tx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

tx_hash, receipt = send_tx(tx_params, wallet_private_key)


Transaction sent: 54edb31eff7fa8fdb9aaa755e0da0351d3cf9c5a780dca5e64c34dbda420fb35  ; Transaction Status: success


## Price back

In [ ]:
USTUSD_in_amount = int(116000.907363250264747423 * 10**18)

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        USTUSD_in_amount,  # amount in, (116000.907363250264747423 USTUSD with 18 decimals)
        0,
        [   
            USTUSD_address,
            500,
            USDT_address
        ],
    ).build(2**256 - 1)

tx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }
tx_hash, receipt = send_tx(tx_params, wallet_private_key)


Hash of universal router swap transaction :  0x1e6f6c1e17ac97a4b9ebbd3e08dc80527152c98753ddb9b5865a40e3f5f2be62
Swap executed successfully!
